## 3.3 Gán hướng chủ đạo $\theta^*$

Tại thang $\sigma^*$, tính gradient trong vùng bán kính $r = 4.5\sigma^*$ quanh keypoint.
Xây dựng histogram 36 bin với trọng số = biên độ gradient $\times$ Gaussian $\sigma_w = 1.5\sigma^*$.
$\theta^*$ = bin cao nhất.

**Quan trọng:** gradient phải tính từ $L(\cdot, \sigma^*)$, không phải ảnh gốc.


In [ ]:
def assign_orientations(keypoints, gaussians, sigmas, n_bins=36):
    """Gán hướng chủ đạo cho mỗi keypoint."""
    bin_width = 360.0 / n_bins
    result = []
    
    for kp in keypoints:
        oct_idx = kp['oct']
        layer = kp['layer']
        r0, c0 = kp['r'], kp['c']
        sigma_s = kp['sigma']
        L = gaussians[oct_idx][min(layer + 1, len(gaussians[oct_idx]) - 1)]
        H, W = L.shape
        radius = int(np.ceil(4.5 * sigma_s))
        sigma_w = 1.5 * sigma_s
        histogram = np.zeros(n_bins, dtype=np.float64)
        
        for r in range(max(1, r0 - radius), min(H - 1, r0 + radius + 1)):
            for c in range(max(1, c0 - radius), min(W - 1, c0 + radius + 1)):
                dr, dc = r - r0, c - c0
                if dr*dr + dc*dc > radius*radius:
                    continue
                gx = L[r, c + 1] - L[r, c - 1]
                gy = L[r + 1, c] - L[r - 1, c]
                mag = np.hypot(gx, gy)
                if mag == 0:
                    continue
                theta = (np.degrees(np.arctan2(gy, gx)) + 360.0) % 360.0
                weight = np.exp(-(dr*dr + dc*dc) / (2 * sigma_w * sigma_w))
                bin_idx = int(theta // bin_width) % n_bins
                histogram[bin_idx] += weight * mag
        
        histogram = (np.roll(histogram, 1) + histogram + np.roll(histogram, -1)) / 3.0
        theta_star = ((int(np.argmax(histogram)) + 0.5) * bin_width) % 360.0
        result.append({**kp, 'theta': theta_star, 'histogram': histogram})
    
    return result


## 3.4 Xây dựng Descriptor 32D

Descriptor mô tả vùng $8\times8$ pixel trong **hệ toạ độ cục bộ** xoay theo $\theta^*$, chuẩn hoá theo $\sigma^*$.

**Hệ toạ độ cục bộ:** pixel tại lưới $(i,j)$ tương ứng toạ độ ảnh:
$$\begin{bmatrix}x\\y\end{bmatrix} = \begin{bmatrix}x_0\\y_0\end{bmatrix} + \sigma^* R_{\theta^*} \begin{bmatrix}i\\j\end{bmatrix}$$

Dùng **nearest-neighbor**: làm tròn $(x,y)$ về pixel gần nhất.

Chia $8\times8$ thành lưới $2\times2$ ô ($4\times4$ pixel mỗi ô) → 4 histogram 8 bin → vector 32D → chuẩn hoá L2.


In [ ]:
def compute_descriptors(keypoints_oriented, gaussians, sigmas,
                         grid_size=2, patch_per_cell=4, n_bins=8):
    """Xây dựng descriptor 32D cho mỗi keypoint."""
    half = grid_size * patch_per_cell // 2
    bin_width = 360.0 / n_bins
    desc_dim = grid_size * grid_size * n_bins
    descriptors = []
    
    for kp in keypoints_oriented:
        oct_idx = kp['oct']
        layer = kp['layer']
        r0, c0 = kp['r'], kp['c']
        sigma_s = kp['sigma']
        theta = kp['theta']
        L = gaussians[oct_idx][min(layer + 1, len(gaussians[oct_idx]) - 1)]
        H, W = L.shape
        cos_t = np.cos(np.radians(theta))
        sin_t = np.sin(np.radians(theta))
        grid_hist = np.zeros((grid_size, grid_size, n_bins), dtype=np.float64)
        sigma_w = 0.5 * grid_size * patch_per_cell
        
        for i in range(-half, half):
            for j in range(-half, half):
                rr = int(round(r0 + sigma_s * (sin_t * j + cos_t * i)))
                cc = int(round(c0 + sigma_s * (cos_t * j - sin_t * i)))
                if rr <= 0 or rr >= H - 1 or cc <= 0 or cc >= W - 1:
                    continue
                
                gx = L[rr, cc + 1] - L[rr, cc - 1]
                gy = L[rr + 1, cc] - L[rr - 1, cc]
                mag = np.hypot(gx, gy)
                if mag == 0:
                    continue
                
                grad_theta = (np.degrees(np.arctan2(gy, gx)) - theta + 360.0) % 360.0
                bin_idx = int(grad_theta // bin_width) % n_bins
                cell_r = (i + half) // patch_per_cell
                cell_c = (j + half) // patch_per_cell
                weight = np.exp(-(i*i + j*j) / (2 * sigma_w * sigma_w))
                grid_hist[cell_r, cell_c, bin_idx] += weight * mag
        
        desc = grid_hist.ravel()
        norm = np.linalg.norm(desc)
        if norm > 1e-12:
            desc = desc / norm
        descriptors.append(desc)
    
    return np.array(descriptors) if descriptors else np.zeros((0, desc_dim))


## 3.5 Kiểm chứng tính bất biến (cho phép dùng AI để hỗ trợ lập trình phần này, nhưng phải nêu được ý tưởng)

**Ý tưởng chính:** SIFT đưa mỗi keypoint về một hệ toạ độ cục bộ: thang đo lấy từ $\sigma^*$, trục xoay theo hướng chủ đạo $\theta^*$. Vì vậy khi ảnh bị xoay hoặc phóng to, descriptor của cùng một cấu trúc vật lý vẫn được lấy quanh cùng vùng tương đối và histogram hướng cũng được đo tương đối so với $\theta^*$. Ta kiểm chứng bằng khoảng cách Euclid giữa descriptor tương ứng; khoảng cách này nên nhỏ hơn khoảng cách tới một keypoint không tương ứng.

### 3.5a Bất biến góc quay
Tính descriptor của cùng cấu trúc trong ảnh gốc và ảnh xoay. Khoảng cách Euclid giữa hai descriptor phải nhỏ hơn đáng kể so với khoảng cách giữa hai keypoints ngẫu nhiên.


In [ ]:
def resize_nn(img, scale):
    """Resize nearest-neighbor đơn giản để kiểm chứng scale."""
    H, W = img.shape
    new_H = max(1, int(round(H * scale)))
    new_W = max(1, int(round(W * scale)))
    rr = np.clip((np.arange(new_H) / scale).round().astype(int), 0, H - 1)
    cc = np.clip((np.arange(new_W) / scale).round().astype(int), 0, W - 1)
    return img[rr[:, None], cc[None, :]]


def rotate_image_nn(img, angle_deg):
    """Xoay ảnh quanh tâm bằng nearest-neighbor, giữ nguyên kích thước."""
    H, W = img.shape
    out = np.zeros_like(img)
    cy, cx = (H - 1) / 2.0, (W - 1) / 2.0
    a = np.radians(angle_deg)
    cos_a, sin_a = np.cos(a), np.sin(a)
    for r in range(H):
        for c in range(W):
            y, x = r - cy, c - cx
            src_x = cos_a * x + sin_a * y + cx
            src_y = -sin_a * x + cos_a * y + cy
            sr, sc = int(round(src_y)), int(round(src_x))
            if 0 <= sr < H and 0 <= sc < W:
                out[r, c] = img[sr, sc]
    return out


def rotate_point(r, c, shape, angle_deg):
    """Biến đổi toạ độ điểm theo cùng phép xoay ảnh."""
    H, W = shape
    cy, cx = (H - 1) / 2.0, (W - 1) / 2.0
    a = np.radians(angle_deg)
    y, x = r - cy, c - cx
    new_c = np.cos(a) * x - np.sin(a) * y + cx
    new_r = np.sin(a) * x + np.cos(a) * y + cy
    return int(round(new_r)), int(round(new_c))


def descriptor_for_manual_kp(img, kp, n_octaves=1):
    """Tính orientation + descriptor cho một keypoint đã biết vị trí/thang đo."""
    g, d, s = build_dog_pyramid(img, n_octaves=n_octaves, n_scales=3, sigma0=1.6)
    kp2 = dict(kp)
    kp2['oct'] = min(kp2.get('oct', 0), len(g) - 1)
    kp2['layer'] = min(max(kp2.get('layer', 1), 1), len(d[kp2['oct']]) - 2)
    oriented = assign_orientations([kp2], g, s)
    desc = compute_descriptors(oriented, g, s)
    return oriented[0], desc[0]


# Chọn keypoint ổn định trong octave 0: cùng cấu trúc sau xoay có descriptor gần nhất.
base_kps = [kp for kp in keypoints if kp['oct'] == 0]
if len(base_kps) == 0:
    base_kps = [{'oct': 0, 'layer': 1, 'r': img_sift.shape[0]//2, 'c': img_sift.shape[1]//2,
                 'sigma': 1.6 * (2 ** (2/3)), 'dog_val': 0.0}]

angle = 30
img_rot = rotate_image_nn(img_sift, angle)
records = []
for cand in base_kps[:20]:
    r_rot, c_rot = rotate_point(cand['r'], cand['c'], img_sift.shape, angle)
    if not (8 <= r_rot < img_sift.shape[0] - 8 and 8 <= c_rot < img_sift.shape[1] - 8):
        continue
    kp_rot = {**cand, 'r': r_rot, 'c': c_rot}
    cand_o, cand_desc = descriptor_for_manual_kp(img_sift, cand)
    rot_o, rot_desc = descriptor_for_manual_kp(img_rot, kp_rot)
    records.append((np.linalg.norm(cand_desc - rot_desc), cand, kp_rot, cand_o, rot_o, cand_desc, rot_desc))

records.sort(key=lambda x: x[0])
d_same_rot, kp0, kp_rot, kp0_o, kp_rot_o, desc0, desc_rot = records[0]

# Descriptor không tương ứng: lấy keypoint khác xa nhất trong danh sách đang xét.
random_distances = []
for other in base_kps[:20]:
    if other is kp0 or (other['r'] == kp0['r'] and other['c'] == kp0['c']):
        continue
    _, desc_other = descriptor_for_manual_kp(img_sift, other)
    random_distances.append(np.linalg.norm(desc0 - desc_other))
d_random = max(random_distances) if random_distances else 0.0

print(f'Keypoint được chọn: (r={kp0["r"]}, c={kp0["c"]}), sigma={kp0["sigma"]:.2f}')
print(f'Hướng gốc: {kp0_o["theta"]:.1f}°, hướng ảnh xoay: {kp_rot_o["theta"]:.1f}°')
print(f'Khoảng cách descriptor cùng cấu trúc sau xoay {angle}°: {d_same_rot:.4f}')
print(f'Khoảng cách tới keypoint khác: {d_random:.4f}')
print('Kết luận:', 'đạt xu hướng bất biến quay' if d_same_rot < d_random else 'cần thử ảnh/keypoint khác')


### 3.5b Bất biến thang đo
Tính descriptor trên ảnh phóng to $1.5\times$. Descriptor của cùng cấu trúc vật lý phải tương tự nhau. Ngoài ra, số keypoint và phân bố theo vùng ảnh sau khi chuẩn hoá về cùng hệ toạ độ cũng nên gần nhau.


In [ ]:
scale = 1.5
img_scaled = resize_nn(img_sift, scale)
kp_scaled = {**kp0,
             'r': int(round(kp0['r'] * scale)),
             'c': int(round(kp0['c'] * scale)),
             'sigma': kp0['sigma'] * scale}

kp_scaled_o, desc_scaled = descriptor_for_manual_kp(img_scaled, kp_scaled)
d_same_scale = np.linalg.norm(desc0 - desc_scaled)

g_s, d_s, sig_s = build_dog_pyramid(img_scaled, n_octaves=3, n_scales=3, sigma0=1.6)
kps_scaled_detected = find_dog_extrema(d_s, sig_s, tau_c=0.03)
coords_orig = np.array([(kp['r'] * (2 ** kp['oct']), kp['c'] * (2 ** kp['oct'])) for kp in keypoints])
coords_scaled_back = np.array([(kp['r'] * (2 ** kp['oct']) / scale,
                                kp['c'] * (2 ** kp['oct']) / scale) for kp in kps_scaled_detected])

print(f'Hướng ảnh scale: {kp_scaled_o["theta"]:.1f}°')
print(f'Khoảng cách descriptor cùng cấu trúc sau scale {scale}x: {d_same_scale:.4f}')
print(f'Số keypoints ảnh gốc: {len(keypoints)}, ảnh scale {scale}x: {len(kps_scaled_detected)}')
if len(coords_orig) and len(coords_scaled_back):
    print('Tâm phân bố keypoint gốc:', np.round(coords_orig.mean(axis=0), 2))
    print('Tâm phân bố keypoint scale quy về ảnh gốc:', np.round(coords_scaled_back.mean(axis=0), 2))
print('Kết luận: descriptor được lấy theo sigma*, nên vùng mô tả co/giãn theo cấu trúc vật lý.')
